In [27]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

In [28]:
SEED = 1990
df_prototype = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/prototype.csv")
df_test = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/test.csv")
pkl_file_path = '/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/features/I3D_features.pkl'
df_features = pd.read_pickle(pkl_file_path)
training_matrix = {}


le = LabelEncoder()
df_prototype["encoded_category"] = le.fit_transform(df_prototype["category"].astype(str))

rows = []
for video_id, enc_cat in zip(df_prototype["uid"].values, df_prototype["encoded_category"].values):
    tensor = (
        df_features.loc[df_features["id"] == video_id, "I3D_features"]
        .iloc[0]
        .squeeze()
    )
    rows.append((video_id, tensor, enc_cat))

df_train = pd.DataFrame(rows, columns=["uid", "I3D_features", "encoded_category"])
df_train.head()
df_train.shape

(4765, 3)

In [29]:


# ---------- Gloss normalization ----------
def normalize_gloss(g):
    return " ".join(str(g).strip().upper().split())

# df_prototype, df_test, df_features already loaded by you
df_prototype = df_prototype.copy()
df_test = df_test.copy()

df_prototype["gloss_norm"] = df_prototype["gloss"].apply(normalize_gloss)
df_test["gloss_norm"] = df_test["gloss"].apply(normalize_gloss)

# Encode gloss using prototype vocabulary
gloss_le = LabelEncoder()
df_prototype["encoded_gloss"] = gloss_le.fit_transform(df_prototype["gloss_norm"])

proto_glosses = set(df_prototype["gloss_norm"])
df_test_seen = df_test[df_test["gloss_norm"].isin(proto_glosses)].copy()
df_test_seen["encoded_gloss"] = gloss_le.transform(df_test_seen["gloss_norm"])

print("Dropped test rows (unseen gloss):", len(df_test) - len(df_test_seen))
print("Unique glosses in prototype:", df_prototype["gloss_norm"].nunique())
print("Unique glosses in test_seen:", df_test_seen["gloss_norm"].nunique())

# Build proto df with features
proto_rows = []
for uid, eg in zip(df_prototype["uid"].values, df_prototype["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    proto_rows.append((uid, tensor, eg))
df_proto_full = pd.DataFrame(proto_rows, columns=["uid","I3D_features","encoded_gloss"])

# Build test df with features
test_rows = []
for uid, eg in zip(df_test_seen["uid"].values, df_test_seen["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    test_rows.append((uid, tensor, eg))
df_test_full  = pd.DataFrame(test_rows,  columns=["uid","I3D_features","encoded_gloss"])

print("PROTO shape:", df_proto_full.shape, "TEST shape:", df_test_full.shape)




Dropped test rows (unseen gloss): 0
Unique glosses in prototype: 4765
Unique glosses in test_seen: 1363
PROTO shape: (4765, 3) TEST shape: (2285, 3)


In [30]:
def pad_features(features, T_max=51):
    D = 1024
    T = features.shape[1]  # safer than len(features[0])

    if T > T_max:
        features = features[:, :T_max]
        T = T_max

    padded = np.zeros((D, T_max), dtype=np.float32)
    padded[:, :T] = features[:, :T]

    mask = np.zeros(T_max, dtype=np.float32)
    mask[:T] = 1.0

    return padded, mask, T


In [31]:
class DFVideoDataset(Dataset):
    def __init__(self, df, label_col="encoded_category"):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        x = self.df.loc[idx, "I3D_features"]
        x = np.array(x)

        # MINIMAL FIX: handle accidental (T,1024)
        if x.ndim == 2 and x.shape[0] != 1024 and x.shape[1] == 1024:
            x = x.T

        assert x.ndim == 2 and x.shape[0] == 1024, f"bad shape at idx={idx}: {x.shape}"
        y = self.df.loc[idx, self.label_col]
        return x, y


def collate_fn(batch, T_max=51):
    batch_size = len(batch)
    features_list = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    padded_batch = np.zeros((batch_size, 1024, T_max), dtype=np.float32)
    mask_batch   = np.zeros((batch_size, T_max), dtype=np.float32)

    for i, feat in enumerate(features_list):
        padded, mask, T = pad_features(feat, T_max=T_max)
        padded_batch[i] = padded
        mask_batch[i] = mask

    return (
        torch.from_numpy(padded_batch),
        torch.from_numpy(mask_batch),
        torch.tensor(labels, dtype=torch.long),
    )

from sklearn.model_selection import train_test_split

# -------------------------
# Train / Val split (CATEGORY)
# -------------------------
counts = df_train["encoded_category"].value_counts()
ok_cats = counts[counts >= 2].index
df_ok = df_train[df_train["encoded_category"].isin(ok_cats)].copy()

df_tr, df_val = train_test_split(
    df_ok,
    test_size=0.2,
    random_state=SEED,
    stratify=df_ok["encoded_category"]
)

# -------------------------
# Build datasets
# -------------------------
ds_tr = DFVideoDataset(df_tr, label_col="encoded_category")
ds_val = DFVideoDataset(df_val, label_col="encoded_category")

train_loader = DataLoader(
    ds_tr, batch_size=8, shuffle=True, collate_fn=collate_fn
)

# -------------------------
# k-shot validation split
# -------------------------
k = 5
vc = df_val["encoded_category"].value_counts()
eligible = vc[vc >= (k + 1)].index
df_val_k = df_val[df_val["encoded_category"].isin(eligible)]

df_val_support = (
    df_val_k
    .groupby("encoded_category", group_keys=False)
    .sample(n=k, random_state=SEED)
)
df_val_query = df_val_k.drop(index=df_val_support.index)

ds_val_support = DFVideoDataset(df_val_support, label_col="encoded_category")
ds_val_query   = DFVideoDataset(df_val_query,   label_col="encoded_category")

val_support_loader = DataLoader(
    ds_val_support, batch_size=32, shuffle=False, collate_fn=collate_fn
)
val_query_loader = DataLoader(
    ds_val_query, batch_size=32, shuffle=False, collate_fn=collate_fn
)

proto_loader = DataLoader(
    DFVideoDataset(df_proto_full, label_col="encoded_gloss"),
    batch_size=32, shuffle=False, collate_fn=collate_fn
)

test_loader = DataLoader(
    DFVideoDataset(df_test_full, label_col="encoded_gloss"),
    batch_size=32, shuffle=False, collate_fn=collate_fn
)


In [ ]:
@torch.no_grad()
def build_proto_matrix(encoder, proto_loader, device):
    encoder.eval()
    Z = []
    Y = []
    for x, m, y in proto_loader:
        x = x.to(device); m = m.to(device); y = y.to(device)
        z = encoder(x, m)               # (B,d) already normalized in your encoder
        Z.append(z)
        Y.append(y)
    Z = torch.cat(Z, dim=0)            # (P,d)
    Y = torch.cat(Y, dim=0)            # (P,)
    return Z, Y

def center_embeddings(Zp, Zq):
    mu = Zp.mean(dim=0, keepdim=True)
    Zp_c = F.normalize(Zp - mu, p=2, dim=1)
    Zq_c = F.normalize(Zq - mu, p=2, dim=1)
    return Zp_c, Zq_c

def two_view_time_aug(x, m, crop_min=0.6, drop_p=0.1):
    """
    x: (B,1024,T), m: (B,T) float {0,1}
    returns two views (x1,m1,x2,m2)
    """
    B, D, T = x.shape
    device = x.device

    def make_view():
        xv = x.clone()
        mv = m.clone()

        for i in range(B):
            valid_idx = torch.where(mv[i] > 0)[0]
            if len(valid_idx) <= 2:
                continue
            L = int(len(valid_idx))
            cL = max(2, int(L * (crop_min + (1.0 - crop_min) * torch.rand(1, device=device).item())))
            if cL >= L:
                continue
            start = int(torch.randint(0, L - cL + 1, (1,), device=device).item())
            keep = valid_idx[start:start+cL]

            mask_new = torch.zeros_like(mv[i])
            mask_new[keep] = 1.0
            mv[i] = mv[i] * mask_new
            xv[i] = xv[i] * mv[i].view(1, -1)

        if drop_p > 0:
            drop = (torch.rand(B, T, device=device) < drop_p).float()
            mv = mv * (1.0 - drop)
            xv = xv * mv.view(B, 1, T)

        return xv, mv

    x1, m1 = make_view()
    x2, m2 = make_view()
    return x1, m1, x2, m2

@torch.no_grad()
def gloss_retrieval_centered_multiview_max(
    encoder, test_loader, Zp, Yp,
    device, num_views=5, crop_min=0.6, drop_p=0.1, topk=(1,5,10)
):
    """
    Prototypes: Zp (P,d), Yp (P,) where each prototype corresponds to one gloss video.
    For each test sample, compute num_views augmented embeddings, take max sim per prototype.
    Uses centered cosine.
    """
    encoder.eval()
    Zp = Zp.to(device); Yp = Yp.to(device)

    # Center prototypes once
    Zp_c, _dummy = two_view_time_aug(Zp, Zp)  # (P,d)

    hits = {k: 0 for k in topk}
    total = 0

    for x, m, y in test_loader:
        x = x.to(device); m = m.to(device); y = y.to(device)
        B = x.size(0)

        # Collect similarity matrices for multiple views, then max over views
        sims_best = None

        for _ in range(num_views):
            # Make one view by using one side of augmentation
            x1, m1, _, _ = two_view_time_aug(x, m, crop_min=crop_min, drop_p=drop_p)
            z = encoder(x1, m1)  # (B,d)

            # Center queries using prototype mean
            mu = Zp.mean(dim=0, keepdim=True)
            zc = F.normalize(z - mu, p=2, dim=1)      # (B,d)

            sims = zc @ Zp_c.t()  # (B,P)

            sims_best = sims if sims_best is None else torch.maximum(sims_best, sims)

        # Pred indices from sims_best
        total += B

        # True prototype index = where Yp == y (but note: many prototypes share unique label only once here)
        # Because 1 sample per gloss, each gloss label maps to exactly one prototype row.
        # Build a mapping from gloss label -> proto row index
        # We'll do it once lazily
        if total == B:
            y_to_idx = {int(lbl.item()): i for i, lbl in enumerate(Yp)}

        true_idx = torch.tensor([y_to_idx[int(t.item())] for t in y], device=device)

        for k in topk:
            pred = sims_best.topk(k, dim=1).indices
            hits[k] += (pred == true_idx.unsqueeze(1)).any(dim=1).sum().item()

    return {k: hits[k]/max(total,1) for k in topk}


# -------------------------
# Utility: build prototypes (from a loader)
# -------------------------
@torch.no_grad()
def compute_class_prototypes(encoder, loader, device):
    """
    Builds mean embedding per class from samples in loader.
    Returns:
        proto_mat: (C, d) L2-normalized
    """
    encoder.eval()

    sums = {}
    counts = {}

    for x, m, y in loader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        z = encoder(x, m)  # (B, d)

        for i in range(z.size(0)):
            cls = int(y[i].item())
            if cls not in sums:
                sums[cls] = z[i].detach().clone()
                counts[cls] = 1
            else:
                sums[cls] += z[i]
                counts[cls] += 1

    # Build ordered prototype matrix
    classes = sorted(sums.keys())
    proto = []
    for c in classes:
        p = sums[c] / max(counts[c], 1)
        p = F.normalize(p, p=2, dim=-1)
        proto.append(p)

    proto_mat = torch.stack(proto, dim=0)  # (C, d)
    return classes, proto_mat


# -------------------------
# Utility: retrieval evaluation (Top-1/5/10)
# -------------------------
@torch.no_grad()
def retrieval_eval(encoder, query_loader, proto_classes, proto_mat, device, topk=(1,5,10)):
    """
    query embeddings vs prototype matrix by cosine similarity.
    """
    encoder.eval()
    proto_mat = proto_mat.to(device)  # (C, d)

    topk_hits = {k: 0 for k in topk}
    total = 0

    for x, m, y in query_loader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        z = encoder(x, m)  # (B, d)

        # cosine sim because vectors are L2-normalized: sim = z @ proto.T
        sims = z @ proto_mat.t()  # (B, C)
        total += z.size(0)

        # Map true labels to prototype indices
        # proto_classes is sorted list of class ids used in proto_mat
        # We'll make a dict for quick lookup (small C=57 so fine)
        idx_map = {c:i for i,c in enumerate(proto_classes)}
        true_idx = torch.tensor([idx_map[int(t.item())] for t in y], device=device)

        # compute top-k
        for k in topk:
            pred_idx = sims.topk(k, dim=1).indices  # (B, k)
            hit = (pred_idx == true_idx.unsqueeze(1)).any(dim=1).sum().item()
            topk_hits[k] += hit

    accs = {k: topk_hits[k]/max(total,1) for k in topk}
    return accs




In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedAttnEncoder(nn.Module):
    """
    Input:
        x: (B, 1024, T)
        mask: (B, T) float {0,1}
    Output:
        z: (B, d) L2-normalized
    """
    def __init__(self, d=256, use_layernorm=True, dropout=0.05):
        super().__init__()
        self.d = d
        self.ln = nn.LayerNorm(1024) if use_layernorm else None

        self.proj = nn.Linear(1024, d)
        self.attn = nn.Linear(d, 1)

        # NEW: pooling fusion head (2d -> d)
        self.pool_fuse = nn.Linear(2 * d, d)

        self.drop = nn.Dropout(p=dropout)

    def masked_mean(self, h, mask):
        # h: (B, T, d), mask: (B, T)
        w = mask.unsqueeze(-1)  # (B,T,1)
        s = (h * w).sum(dim=1)  # (B,d)
        denom = w.sum(dim=1).clamp(min=1e-6)  # (B,1)
        return s / denom

    def masked_max(self, h, mask):
        # set padded positions to -inf before max
        neg_inf = torch.finfo(h.dtype).min
        h2 = h.masked_fill(mask.unsqueeze(-1) == 0, neg_inf)
        return h2.max(dim=1).values

    def forward(self, x, mask):
        # x: (B,1024,T) -> (B,T,1024)
        x = x.transpose(1, 2)

        if self.ln is not None:
            x = self.ln(x)

        h = self.proj(x)           # (B,T,d)
        h = self.drop(h)

        # attention weighted sum (your existing idea)
        scores = self.attn(torch.tanh(h)).squeeze(-1)      # (B,T)
        scores = scores.masked_fill(mask == 0, -1e9)
        alpha = F.softmax(scores, dim=1)                   # (B,T)
        z_attn = torch.sum(h * alpha.unsqueeze(-1), dim=1) # (B,d)

        # NEW: mean+max pooling summary
        z_mean = self.masked_mean(h, mask)  # (B,d)
        z_max  = self.masked_max(h, mask)   # (B,d)

        z_pool = torch.cat([z_mean, z_max], dim=-1)        # (B,2d)
        z_pool = self.pool_fuse(z_pool)                    # (B,d)

        # combine (simple average). You can later try weighted sum.
        z = 0.5 * z_attn + 0.5 * z_pool

        z = F.normalize(z, p=2, dim=-1)
        return z
class EncoderWithHead(nn.Module):
    def __init__(self, num_classes, d=256, dropout=0.05):
        super().__init__()
        self.encoder = MaskedAttnEncoder(d=d, dropout=dropout)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x, mask):
        z = self.encoder(x, mask)      # (B, d)
        logits = self.head(z)          # (B, num_classes)
        return logits, z


In [36]:
# -------------------------
# Stage-1 training (CE) with a one-time sanity debug print
# Paste this whole cell and run.
# -------------------------
from tqdm.auto import tqdm
import copy
import torch
import torch.nn as nn

def train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps",
    debug_first_batch=True,
):
    model = EncoderWithHead(num_classes=num_classes, d=256).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_top1 = -1.0
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    did_debug = False  # <-- runs sanity print exactly once

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for bidx, (x, m, y) in enumerate(pbar):
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            logits, z = model(x, m)
            loss = criterion(logits, y)

            # ---- SANITY DEBUG (first batch only) ----
            if debug_first_batch and (not did_debug) and (ep == 1) and (bidx == 0):
                print("\n=== SANITY CHECK (first batch) ===")
                print("num_classes:", num_classes)
                print("x:", tuple(x.shape), x.dtype)
                print("m:", tuple(m.shape), m.dtype, "mask_sum[0]:", float(m[0].sum().item()))
                print("y:", tuple(y.shape), y.dtype, "min/max:", int(y.min().item()), int(y.max().item()))
                print("logits:", tuple(logits.shape), logits.dtype, "logits[0,:5]:", logits[0, :5].detach().cpu())
                print("z:", tuple(z.shape), z.dtype, "z_norm[0]:", float(z[0].norm().item()))
                assert logits.ndim == 2, f"logits.ndim={logits.ndim}"
                assert logits.size(1) == num_classes, f"logits.size(1)={logits.size(1)} != num_classes={num_classes}"
                assert y.dtype == torch.long, f"y.dtype={y.dtype} (must be torch.long for CE)"
                did_debug = True
                print("✅ Sanity checks passed.\n")

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            total_correct += (logits.argmax(dim=1) == y).sum().item()

            pbar.set_postfix(
                loss=total_loss / max(total_seen, 1),
                acc=total_correct / max(total_seen, 1),
            )

        train_loss = total_loss / max(total_seen, 1)
        train_acc  = total_correct / max(total_seen, 1)

        # retrieval validation (category)
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        top1 = float(val_accs.get(1, 0.0))

        print(
            f"Epoch {ep:02d} | "
            f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

        # early stopping on Val Top1
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model


# ---- Call example ----
num_classes = df_train["encoded_category"].nunique()
device = "cuda" if torch.cuda.is_available() else "mps"

model = train_stage1(
    train_loader=train_loader,
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    num_classes=num_classes,
    epochs=30,              # keep small for debugging first
    lr=1e-4,
    weight_decay=0.05,
    patience=5,
    min_delta=1e-4,
    device=device,
    debug_first_batch=True
)


Epoch 1/30:   0%|          | 0/477 [00:00<?, ?it/s]

Epoch 1/30:   5%|▌         | 24/477 [00:00<00:03, 124.76it/s, acc=0.0481, loss=4.05]


=== SANITY CHECK (first batch) ===
num_classes: 58
x: (8, 1024, 51) torch.float32
m: (8, 51) torch.float32 mask_sum[0]: 22.0
y: (8,) torch.int64 min/max: 1 56
logits: (8, 58) torch.float32 logits[0,:5]: tensor([ 0.0905, -0.0107,  0.0154, -0.0277, -0.0165])
z: (8, 256) torch.float32 z_norm[0]: 1.0
✅ Sanity checks passed.



Epoch 01 | TrainLoss=3.9346 TrainAcc=0.1249 | ValRetr Top1=0.1182 Top5=0.2657 Top10=0.3635


Epoch 02 | TrainLoss=3.8019 TrainAcc=0.1588 | ValRetr Top1=0.1314 Top5=0.2788 Top10=0.3956


...

Epoch 09 | TrainLoss=3.3431 TrainAcc=0.2314 | ValRetr Top1=0.1401 Top5=0.3182 Top10=0.4628


Epoch 10 | TrainLoss=3.2910 TrainAcc=0.2414 | ValRetr Top1=0.1489 Top5=0.3285 Top10=0.4774


Epoch 11 | TrainLoss=3.2382 TrainAcc=0.2495 | ValRetr Top1=0.1489 Top5=0.3182 Top10=0.4876


Epoch 12 | TrainLoss=3.1837 TrainAcc=0.2663 | ValRetr Top1=0.1562 Top5=0.3328 Top10=0.4891


Epoch 13 | TrainLoss=3.1301 TrainAcc=0.2737 | ValRetr Top1=0.1577 Top5=0.3270 Top10=0.5066
Early stopping at epoch 13 (best Top1=0.1591 at epoch 8).
Restored best model from epoch 8 (Top1=0.1591).


In [37]:
Zp, Yp = build_proto_matrix(model.encoder, proto_loader, device=device)

accs = gloss_retrieval_centered_multiview_max(
    model.encoder,
    test_loader,
    Zp, Yp,
    device=device,
    num_views=5,     # freeze this
    crop_min=0.6,    # freeze this
    drop_p=0.1,      # freeze this
    topk=(1,5,10)
)

print("\nFINAL TEST (Gloss Retrieval: Centered Cosine + MultiViewMax)")
print("Top-1 :", round(accs[1], 4))
print("Top-5 :", round(accs[5], 4))
print("Top-10:", round(accs[10], 4))



FINAL TEST (Gloss Retrieval: Centered Cosine + MultiViewMax)
Top-1 : 0.137
Top-5 : 0.1694
Top-10: 0.1886
